### Krocket – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024), 1. Runde, Aufgabe 4*

Laura findet beim Spielen im Garten einige aufgestellte Tore eines Krocketspiels. Ihr Vater ruft aus der Küche, dass er es mit nur zwei Schlägen
geschafft hat, den Ball durch alle Tore in der richtigen Reihenfolge zu stoßen. Das möchte Laura verbessern, wobei sie sich allerdings erlaubt, von einem
selbst gewähltem Punkt aus abzuschlagen. Aber geht das überhaupt?

#### Aufgabe 4

Hilf Laura, indem du ein Programm schreibst, das überprüft, ob alle Tore in der richtigen Reihenfolge mit nur einem Schlag durchquert werden können. Falls ja,
soll dein Programm ein entsprechendes Paar aus Startpunkt und Schlagrichtung ausgeben.

Der Ball ist rund und hat einen in der Eingabe angegebenen Radius r. Die Tore sehen von oben wie verschieden lange Geradensegmente aus und sind in der
vorgeschriebenen Reihenfolge jeweils durch ihre zwei Endpunkte dargestellt.

**Hinweis:** Es ist hilfreich, den Radius des Balls in einem ersten Lösungsversuch zu vernachlässigen, das heißt r = 0 anzunehmen.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `krocket1.txt` bis `krocket5.txt`. Die erste Zeile enthält die Zahl `n` der Tore und den Radius `r` des Balls. Danach folgt
für jedes Tor eine Zeile `x1 y1 x2 y2` mit den Koordinaten der beiden Endpunkte, in der vorgeschriebenen Reihenfolge. `krocket3.txt` bis `krocket5.txt` enthalten
mehrere hundert Tore.

```
2 7
10 20 30 40
12 31 11 8
```

Hier stehen zwei Tore im Garten und der Ball hat den Radius 7 cm. Die Tore sind Strecken mit den Endpunkten (10|20) und (30|40) sowie (12|31) und (11|8).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Aus der Aufgabe wird Geometrie:** Der Mittelpunkt des Balls bewegt sich auf einer **Geraden**. Der Ball passt durch ein Tor, wenn die Gerade die Torstrecke schneidet und die
  beiden **Torpfosten** (die Endpunkte) auf verschiedenen Seiten liegen, jeder mit einem Abstand von **mindestens `r`** zur Geraden.
  Wir suchen also eine Gerade, die alle Tore **in der richtigen Reihenfolge** so durchquert. Eine solche Gerade nennen wir **nützlich**.
- **Gerade in Normalform:** Eine Gerade beschreibt man durch einen **Einheitsvektor `n`** senkrecht zu ihr und eine Zahl `c`. Dann ist `vz_abstand(p) = n · p − c` der
  **vorzeichenbehaftete Abstand** eines Punktes `p` von der Geraden: Der Betrag ist der Abstand, das Vorzeichen sagt, auf welcher Seite der Punkt liegt.
- **Schnittpunkt Gerade und Strecke:** Hat ein Tor die Endpunkte `A` und `B` mit den Abständen `sa` und `sb` (mit verschiedenen Vorzeichen), liegt der Schnittpunkt bei
  `A + (B − A) · sa / (sa − sb)`.
- **Reihenfolge entlang der Geraden:** Mit dem Richtungsvektor `t` der Geraden bekommt jeder Punkt auf ihr eine Zahl `p = t · punkt`. Die Reihenfolge der Tore entlang der Geraden ist
  die Reihenfolge dieser Zahlen. Sie muss durchgehend steigen oder durchgehend fallen.
- **Endlich viele Kandidaten:** Unendlich viele Geraden kann man nicht ausprobieren. Aber: Hat man eine nützliche Gerade, kann man sie **parallel verschieben**, bis sie einen Torpfosten im Abstand genau `r` berührt, und dann
  **um den Kreis** um diesen Pfosten drehen, bis sie einen zweiten Pfosten berührt. Sie bleibt dabei nützlich. Es genügt also, Geraden zu prüfen, die zwei Torpfosten **im Abstand genau `r`** berühren
  (Tangenten an zwei Kreise). Für jedes Pfostenpaar sind das höchstens vier.
- **Rechnen mit Kommazahlen:** Beim Berechnen der Geraden entstehen winzige Rundungsfehler. Ein Torpfosten, der **genau** im Abstand `r` sein sollte, kommt dann mit `r − 0,0000000000001` heraus. Darum
  rechnen wir mit einer kleinen **Toleranz**, siehe Schritt 4.

**Nützliche Funktionen und Bibliotheken**
- `math.sqrt`, Tupel `(x, y)`, Listen von Tupeln
- Schleifen mit frühem Abbruch (`return` oder `break`, sobald ein Tor nicht passt)

## Die Idee

1. Die Tore einlesen und eine Funktion bauen, die für eine **gegebene Gerade** prüft, ob sie nützlich ist.
2. Für **jedes Paar von Torpfosten** die Geraden bestimmen, die beide im Abstand `r` berühren.
3. Jede dieser Geraden prüfen. Findet man eine nützliche, hat man die Lösung. Findet man keine, geht es nicht.
4. Aus der Geraden **Startpunkt und Schlagrichtung** machen und das Ergebnis mit den genauen Rundungswerten nachprüfen.

## Schritt 1: Die Tore einlesen

Ein Tor ist ein Paar von Punkten `(A, B)`, ein Punkt ist ein Tupel `(x, y)`. Wir lesen alle Zahlen der Datei: Die ersten beiden sind `n` und `r`, dann folgen je vier Zahlen pro Tor.

In [ ]:
def lies_tore(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = []
        for text in f.read().split():
            zahlen.append(float(text))
    n = int(zahlen[0])
    r = zahlen[1]
    tore = []
    for i in range(n):
        a = (zahlen[2 + 4 * i], zahlen[3 + 4 * i])
        b = (zahlen[4 + 4 * i], zahlen[5 + 4 * i])
        tore.append((a, b))
    return r, tore


r, tore = lies_tore('beispieldaten/krocket1.txt')
print('Radius', r, ', Tore:', len(tore))
print(tore[:3])

Radius 1.0 , Tore: 9
[((10.0, 20.0), (12.0, 3.0)), ((20.0, 20.0), (14.0, 3.0)), ((33.0, 15.0), (23.0, 16.0))]


## Schritt 2: Geraden und Abstände

Eine Gerade speichern wir als Tupel `(nx, ny, c)`: `(nx, ny)` ist der Einheitsvektor senkrecht zur Geraden und `c` die Zahl mit `nx · x + ny · y = c` für alle Punkte `(x, y)` auf der Geraden.

`gerade_durch(p, q)` baut die Gerade durch zwei Punkte: Der Richtungsvektor ist `q − p`. Senkrecht dazu steht `(−dy, dx)`, geteilt durch die Länge, damit er Länge 1 hat. `c` bekommt man, indem man `p`
einsetzt. Die **Richtung** der Geraden ist `t = (−ny, nx)`. Der Punkt mit der Zahl `p` auf der Geraden ist `c · n + p · t`.

`vz_abstand(g, p)` ist der vorzeichenbehaftete Abstand eines Punktes von der Geraden.

In [ ]:
from math import sqrt


def gerade_durch(p, q):
    dx = q[0] - p[0]
    dy = q[1] - p[1]
    laenge = sqrt(dx * dx + dy * dy)
    nx = -dy / laenge
    ny = dx / laenge
    return (nx, ny, nx * p[0] + ny * p[1])


def vz_abstand(g, punkt):
    return g[0] * punkt[0] + g[1] * punkt[1] - g[2]


g = gerade_durch((0, 0), (4, 3))
print('Gerade:', g)
for punkt in [(0, 0), (4, 3), (3, 4), (-3, 4), (4, 0)]:
    print(punkt, 'hat den vorzeichenbehafteten Abstand', round(vz_abstand(g, punkt), 3))

Gerade: (-0.6, 0.8, 0.0)
(0, 0) hat den vorzeichenbehafteten Abstand 0.0
(4, 3) hat den vorzeichenbehafteten Abstand 0.0
(3, 4) hat den vorzeichenbehafteten Abstand 1.4
(-3, 4) hat den vorzeichenbehafteten Abstand 5.0
(4, 0) hat den vorzeichenbehafteten Abstand -2.4


Die Gerade durch `(0, 0)` und `(4, 3)` hat `n = (−0,6; 0,8)`. Die Punkte auf der Geraden haben den Abstand 0. Der Punkt `(3, 4)` liegt auf der einen Seite im Abstand 1,4,
der Punkt `(4, 0)` auf der anderen im Abstand 2,4 (mit negativem Vorzeichen).

## Schritt 3: Ist eine Gerade nützlich?

`pruefe_gerade(g, tore, mindestabstand, reihenfolge)` geht die Tore der Reihe nach durch. Für jedes Tor `(A, B)`:

1. Die vorzeichenbehafteten Abstände `sa` und `sb` berechnen. **Beide Pfosten müssen mindestens den `mindestabstand` haben**: `|sa| ≥ mindestabstand` und `|sb| ≥ mindestabstand`.
2. Die Pfosten müssen auf **verschiedenen Seiten** liegen, also `sa · sb < 0`. Dann schneidet die Gerade die Torstrecke. (Bei Radius 0 darf ein Pfosten auch genau auf der Geraden liegen.
   Liegen beide darauf, ist das Tor selbst auf der Geraden und hat keinen einzelnen Schnittpunkt, dann lassen wir die Gerade weg.)
3. Den **Schnittpunkt** berechnen und seine Zahl `p` entlang der Geraden (`p = t · schnittpunkt`).
4. Die Zahlen `p` müssen **durchgehend steigen oder durchgehend fallen**. Das merken wir uns in `richtung` (`+1` steigend, `−1` fallend). Setzt man `reihenfolge=False`, lassen wir diese Prüfung weg und
   fragen nur, ob die Gerade alle Tore schneidet.

Sobald ein Tor nicht passt, hören wir sofort auf (`None`). Sonst bekommen wir die Richtung und die Liste der `p`-Werte zurück.

In [ ]:
def pruefe_gerade(g, tore, mindestabstand, reihenfolge=True):
    """Ergebnis: None, wenn die Gerade nicht nützlich ist, sonst (richtung, liste der p-Werte der Schnittpunkte)."""
    nx, ny, c = g
    tx = -ny
    ty = nx
    werte = []
    richtung = 0
    for a, b in tore:
        sa = nx * a[0] + ny * a[1] - c
        sb = nx * b[0] + ny * b[1] - c
        if abs(sa) < mindestabstand or abs(sb) < mindestabstand:
            return None                                    # ein Torpfosten ist zu nah an der Geraden
        if sa * sb > 0:
            return None                                    # beide Pfosten auf derselben Seite: kein Schnitt
        if sa == sb:
            return None                                    # beide Pfosten auf der Geraden (nur bei Radius 0): kein einzelner Schnittpunkt
        anteil = sa / (sa - sb)
        x = a[0] + (b[0] - a[0]) * anteil
        y = a[1] + (b[1] - a[1]) * anteil
        p = tx * x + ty * y
        if reihenfolge and len(werte) > 0:
            if richtung == 0:
                if p == werte[-1]:
                    return None
                if p > werte[-1]:
                    richtung = 1
                else:
                    richtung = -1
            elif (p - werte[-1]) * richtung <= 0:
                return None                                # die Reihenfolge stimmt nicht
        werte.append(p)
    if richtung == 0:
        richtung = 1
    return richtung, werte


# Die Gerade aus den Lösungshinweisen für krocket1: Start (-1000, -509.656), Richtung (61000, 31047.9)
r, tore = lies_tore('beispieldaten/krocket1.txt')
start = (-1000, -509.656)
ende = (-1000 + 61000, -509.656 + 31047.9)
g = gerade_durch(start, ende)
ergebnis = pruefe_gerade(g, tore, r)
print('nützlich:', ergebnis is not None, ', Richtung', ergebnis[0])

nützlich: True , Richtung -1


Die Gerade aus den offiziellen Lösungshinweisen ist für `krocket1.txt` nützlich: Alle 9 Tore werden in der richtigen Reihenfolge geschnitten, und jeder Pfosten ist mindestens 1 (so groß ist
der Radius) von ihr entfernt. Die Richtung ist `−1`: Entlang von `t` fällt die Zahl `p`, die Tore kommen also in der Gegenrichtung zu `t` der Reihe nach. Das ist in Ordnung, denn `t` ist nur eine von zwei
Richtungen der Geraden.

## Schritt 4: Die Kandidaten, Tangenten an zwei Kreise

Wir suchen Geraden, die zwei Torpfosten `P` und `Q` **im Abstand genau `r`** berühren. Die Gerade berührt dann die Kreise mit Radius `r` um `P` und `Q`. Es gibt vier Möglichkeiten, wie sich die Gerade
zu den beiden Kreisen stellt:

- **Beide Pfosten auf derselben Seite:** Die Gerade liegt parallel zu `PQ` im Abstand `r`, auf der einen oder der anderen Seite. Das sind zwei Geraden.
- **Pfosten auf verschiedenen Seiten:** Die Gerade kreuzt die Strecke `PQ` in ihrer Mitte. Das geht nur, wenn die Kreise sich nicht berühren (`|PQ| ≥ 2r`). Das sind wieder zwei Geraden.

Rechnerisch: Sei `u` der Einheitsvektor von `P` nach `Q` und `w = (−uy, ux)` senkrecht dazu. Die parallelen Geraden haben die Normale `w` und `c = w · P ± r`.
Für die **gekreuzten** Geraden hat die Normale die Form `a · u + b · w` mit `a = 2r / |PQ|` und `b = ±√(1 − a²)`, und `c = n · P + r`: Dann ist `P` im Abstand `−r` und `Q` im Abstand `+r`.
Für `r = 0` würden alle vier in die Gerade durch `P` und `Q` zusammenfallen. Dank `DELTA` (siehe unten) kommt das in unserer Suche nicht vor.

**Toleranz:** Die Tangenten berühren die Pfosten im Abstand genau `r`, und der Rechner trifft das nur bis auf Rundungsfehler. Prüften wir mit `≥ r`, würden gerade diese Pfosten wegen
`r − 0,000 000 000 000 1` durchfallen. Darum bilden wir die Tangenten für einen **etwas größeren Radius** `r + DELTA` und prüfen mit `r + DELTA / 2`. Das gilt auch für `r = 0`: Dann ist es ein winzig kleiner Ball. Für `DELTA = 0,00001`, einen im Vergleich zu den Maßen der Tore winzigen Wert, hat die Gerade am Ende sicher
einen Mindestabstand `r + DELTA / 2` von allen Pfosten. Das gibt auch Platz, die Ausgabe auf ein paar Nachkommastellen zu runden.

In [ ]:
DELTA = 0.00001


def tangenten(p, q, r):
    """Die Geraden, die beide Punkte im Abstand r berühren (höchstens vier)."""
    dx = q[0] - p[0]
    dy = q[1] - p[1]
    d = sqrt(dx * dx + dy * dy)
    if d == 0:
        return []
    ux = dx / d
    uy = dy / d
    wx = -uy
    wy = ux
    c0 = wx * p[0] + wy * p[1]
    if r == 0:
        return [(wx, wy, c0)]
    gerade = [(wx, wy, c0 + r), (wx, wy, c0 - r)]          # parallel zur Verbindung
    a = 2 * r / d
    if a <= 1:                                             # gekreuzt, nur wenn sich die Kreise nicht überlappen
        b = sqrt(1 - a * a)
        for vorzeichen in [1, -1]:
            nx = a * ux + vorzeichen * b * wx
            ny = a * uy + vorzeichen * b * wy
            gerade.append((nx, ny, nx * p[0] + ny * p[1] + r))
    return gerade


for g in tangenten((0, 0), (10, 0), 2):
    print(g, ': Abstand von (0,0):', round(vz_abstand(g, (0, 0)), 6), ', von (10,0):', round(vz_abstand(g, (10, 0)), 6))

(-0.0, 1.0, 2.0) : Abstand von (0,0): -2.0 , von (10,0): -2.0
(-0.0, 1.0, -2.0) : Abstand von (0,0): 2.0 , von (10,0): 2.0
(0.4, 0.916515138991168, 2.0) : Abstand von (0,0): -2.0 , von (10,0): 2.0
(0.4, -0.916515138991168, 2.0) : Abstand von (0,0): -2.0 , von (10,0): 2.0


Bei den Punkten `(0, 0)` und `(10, 0)` mit `r = 2` gibt es vier Tangenten. Zwei sind waagerecht im Abstand 2 über und unter der Verbindung, bei beiden haben die Punkte denselben Abstand. Bei den
anderen beiden liegen die Punkte auf verschiedenen Seiten, einer im Abstand 2 und der andere im Abstand −2.

## Schritt 5: Alle Paare ausprobieren

Wir sammeln alle Torpfosten in einer Liste und probieren **jedes Paar** aus. Für jedes Paar bilden wir die Tangenten für `r + DELTA` und prüfen sie mit dem Mindestabstand `r + DELTA / 2`. Die erste nützliche Gerade
ist die Lösung. In `geprueft` zählen wir, wie viele Geraden angeschaut wurden.

Bei `n` Toren sind das `2n` Pfosten, `n · (2n − 1)` Paare und höchstens `4 · n · (2n − 1)` Geraden. Für jede müssen wir im schlimmsten Fall alle `n` Tore prüfen, das ergibt einen Aufwand von etwa `n³`. Weil die
meisten Geraden schon am ersten oder zweiten Tor scheitern, ist es in der Praxis viel weniger.

`finde_gerade(tore, r, reihenfolge)` gibt die Gerade, die Richtung, die `p`-Werte der Schnittpunkte und die Zahl der geprüften Geraden zurück, oder `None` und die Zahl.

In [ ]:
def finde_gerade(tore, r, reihenfolge=True):
    pfosten = []
    for a, b in tore:
        pfosten.append(a)
        pfosten.append(b)
    radius = r + DELTA                                     # auch bei r = 0: ein winzig kleiner Ball
    mindest = r + DELTA / 2
    geprueft = 0
    for i in range(len(pfosten)):
        for j in range(i + 1, len(pfosten)):
            for g in tangenten(pfosten[i], pfosten[j], radius):
                geprueft = geprueft + 1
                ergebnis = pruefe_gerade(g, tore, mindest, reihenfolge)
                if ergebnis is not None:
                    return g, ergebnis[0], ergebnis[1], geprueft
    return None, None, None, geprueft


for nr in range(1, 6):
    r, tore = lies_tore('beispieldaten/krocket' + str(nr) + '.txt')
    g, richtung, werte, geprueft = finde_gerade(tore, r)
    if g is None:
        print('krocket' + str(nr) + '.txt:', len(tore), 'Tore, r =', r, '-> keine nützliche Gerade,', geprueft, 'Geraden geprüft')
    else:
        print('krocket' + str(nr) + '.txt:', len(tore), 'Tore, r =', r, '-> nützliche Gerade gefunden nach', geprueft, 'Geraden')

krocket1.txt: 9 Tore, r = 1.0 -> nützliche Gerade gefunden nach 279 Geraden
krocket2.txt: 9 Tore, r = 1.0 -> keine nützliche Gerade, 610 Geraden geprüft
krocket3.txt: 396 Tore, r = 20.0 -> keine nützliche Gerade, 1252944 Geraden geprüft
krocket4.txt: 344 Tore, r = 25.0 -> nützliche Gerade gefunden nach 28993 Geraden
krocket5.txt: 406 Tore, r = 484.0 -> nützliche Gerade gefunden nach 110324 Geraden


Für `krocket1.txt`, `krocket4.txt` und `krocket5.txt` gibt es eine nützliche Gerade, für `krocket2.txt` und `krocket3.txt` nicht. Das stimmt mit den offiziellen Lösungshinweisen überein.
Bei `krocket2` und `krocket3` mussten alle Geraden geprüft werden, denn nur so weiß man, dass es keine gibt.

## Schritt 6: Startpunkt und Schlagrichtung

Aus der Geraden machen wir die verlangte Ausgabe: einen **Startpunkt** und eine **Schlagrichtung** (ein Vektor). Die Zahlen `p` der Schnittpunkte sagen uns, wo die Tore liegen. Der Startpunkt liegt
**1000 Längeneinheiten vor dem ersten Tor** auf der Geraden, der Endpunkt des Schlags 1000 hinter dem letzten. Die Schlagrichtung ist der Vektor vom Start zum Endpunkt. Ist die Richtung der Geraden
`t` fallend (`richtung = −1`), geht der Schlag in die Gegenrichtung.

**Nachprüfen:** `pruefe_schlag` nimmt nur Startpunkt, Schlagrichtung, Tore und den echten Radius `r`, **ohne** die Toleranz, und prüft mit den Zahlen, die tatsächlich ausgegeben werden (auf 6 Nachkommastellen
gerundet), ob der Schlag gültig ist: alle Tore in der Reihenfolge, jeder Pfosten mindestens im Abstand `r`.

In [ ]:
def schlag_aus_gerade(g, richtung, werte):
    nx, ny, c = g
    tx = -ny
    ty = nx
    p_start = werte[0] - richtung * 1000
    p_ende = werte[-1] + richtung * 1000
    start = (c * nx + p_start * tx, c * ny + p_start * ty)
    ziel = (c * nx + p_ende * tx, c * ny + p_ende * ty)
    start = (round(start[0], 6), round(start[1], 6))
    schlagrichtung = (round(ziel[0] - start[0], 6), round(ziel[1] - start[1], 6))
    return start, schlagrichtung


def pruefe_schlag(start, schlagrichtung, tore, r):
    ende = (start[0] + schlagrichtung[0], start[1] + schlagrichtung[1])
    g = gerade_durch(start, ende)
    ergebnis = pruefe_gerade(g, tore, r)
    if ergebnis is None:
        return False
    richtung, werte = ergebnis
    # die Tore müssen in Schlagrichtung kommen: p wächst, wenn t (= (-ny, nx)) mit der Schlagrichtung übereinstimmt
    tx = -g[1]
    ty = g[0]
    in_schlagrichtung = tx * schlagrichtung[0] + ty * schlagrichtung[1] > 0
    return (richtung == 1) == in_schlagrichtung


for nr in [1, 4, 5]:
    r, tore = lies_tore('beispieldaten/krocket' + str(nr) + '.txt')
    g, richtung, werte, geprueft = finde_gerade(tore, r)
    start, schlagrichtung = schlag_aus_gerade(g, richtung, werte)
    print('krocket' + str(nr) + '.txt: Start', start, ', Schlagrichtung', schlagrichtung, ', gültig:', pruefe_schlag(start, schlagrichtung, tore, r))

krocket1.txt: Start (-879.472457, -448.309998) , Schlagrichtung (2009.227856, 1022.660885) , gültig: True
krocket4.txt: Start (-887.265965, -271.160239) , Schlagrichtung (10727.148655, 5394.664677) , gültig: True
krocket5.txt: Start (354.200336, 14931.649033) , Schlagrichtung (49416.620693, -12418.998438) , gültig: True


Zum Beispiel für `krocket1.txt`: Der Ball wird vom Punkt `(−879,472; −448,310)` aus in Richtung `(2009,228; 1022,661)` geschlagen, das ist ein Schlag in Richtung rechts oben. Das ist nicht dieselbe Gerade wie
in den Lösungshinweisen, aber es gibt viele nützliche Geraden, und wir geben die erste aus, die wir finden. Die Prüfung `pruefe_schlag` rechnet mit den ausgegebenen, gerundeten Zahlen nach und akzeptiert alle drei Schläge.
Dort bekommt `pruefe_schlag` übrigens auch die Schläge aus den Lösungshinweisen als gültig bestätigt (das prüft der Generator).

## Schritt 7: Warum es manchmal nicht geht

Bei `krocket2.txt` und `krocket3.txt` gibt es keine Lösung. Warum? Das zeigt eine Frage: Gibt es wenigstens eine Gerade, die **alle Tore schneidet** und von jedem Pfosten Abstand `r` hat, wenn wir die
**Reihenfolge** außer Acht lassen (`reihenfolge=False`)? Und: Wäre es mit einem **kleineren Ball** möglich (`r = 0`, der erste Lösungsversuch der Aufgabe)?

In [ ]:
for nr in [2, 3]:
    r, tore = lies_tore('beispieldaten/krocket' + str(nr) + '.txt')
    g, richtung, werte, geprueft = finde_gerade(tore, r, False)
    ohne_reihenfolge = g is not None
    g0, richtung0, werte0, geprueft0 = finde_gerade(tore, 0, True)
    kleiner_ball = g0 is not None
    print('krocket' + str(nr) + '.txt (r =', r, '): Gerade durch alle Tore, ohne auf die Reihenfolge zu achten:', ohne_reihenfolge,
          ', mit r = 0 und Reihenfolge:', kleiner_ball)

krocket2.txt (r = 1.0 ): Gerade durch alle Tore, ohne auf die Reihenfolge zu achten: False , mit r = 0 und Reihenfolge: False
krocket3.txt (r = 20.0 ): Gerade durch alle Tore, ohne auf die Reihenfolge zu achten: True , mit r = 0 und Reihenfolge: False


Bei `krocket2.txt` gibt es nicht einmal eine Gerade, die alle Tore mit dem Radius `r` durchquert, egal in welcher Reihenfolge. Bei `krocket3.txt` gibt es sie schon: Die Tore lassen sich alle schneiden, aber **nicht in der
vorgeschriebenen Reihenfolge**. Die Tore stehen abwechselnd weit links und weit rechts, und eine Gerade durchquert sie dann in der Reihenfolge von links nach rechts statt im Zickzack. So steht es auch in den
Lösungshinweisen. Auch mit dem Radius 0 gibt es bei beiden Beispielen keinen Schlag, der kleinere Ball ändert also nichts.

## Schritt 8: Alle Beispiele

Zum Schluss alle fünf Beispiele mit dem Ergebnis und dem Schlag, wenn es einen gibt. Jeder Schlag wird mit `pruefe_schlag` nachgeprüft, mit dem echten Radius.

In [ ]:
ergebnisse = {}
for nr in range(1, 6):
    r, tore = lies_tore('beispieldaten/krocket' + str(nr) + '.txt')
    g, richtung, werte, geprueft = finde_gerade(tore, r)
    text = 'krocket' + str(nr) + '.txt (' + str(len(tore)) + ' Tore, r = ' + str(r) + '): '
    if g is None:
        ergebnisse[nr] = None
        print(text + 'KEIN Schlag möglich.')
    else:
        start, schlagrichtung = schlag_aus_gerade(g, richtung, werte)
        ergebnisse[nr] = (start, schlagrichtung)
        print(text + 'Start ' + str(start) + ', Schlagrichtung ' + str(schlagrichtung) + ', gültig: ' + str(pruefe_schlag(start, schlagrichtung, tore, r)))

krocket1.txt (9 Tore, r = 1.0): Start (-879.472457, -448.309998), Schlagrichtung (2009.227856, 1022.660885), gültig: True
krocket2.txt (9 Tore, r = 1.0): KEIN Schlag möglich.
krocket3.txt (396 Tore, r = 20.0): KEIN Schlag möglich.
krocket4.txt (344 Tore, r = 25.0): Start (-887.265965, -271.160239), Schlagrichtung (10727.148655, 5394.664677), gültig: True
krocket5.txt (406 Tore, r = 484.0): Start (354.200336, 14931.649033), Schlagrichtung (49416.620693, -12418.998438), gültig: True


**Ergebnis:** Wie in den offiziellen Lösungshinweisen gibt es für `krocket1`, `krocket4` und `krocket5` einen Schlag, für `krocket2` und `krocket3` nicht. Die Startpunkte und Richtungen können sich von denen in den Hinweisen
unterscheiden (die nennen `(−1000; −509,656)` und `(61000; 31047,9)` für `krocket1`, `(−1000; −323,2)` und `(61000; 30465,4)` für `krocket4` und `(−1000; 15263,1)` und `(61000; −15261,6)` für `krocket5`), denn jede nützliche Gerade ist
richtig. Beachte: Bei `krocket5` ist `r = 484` sehr groß gegenüber den Torabständen, und der Spielraum ist klein. Die Hinweisgerade hat nur 0,007 Abstand mehr als `r` zu allen Pfosten.

## Für Neugierige

**Warum reicht die Tangentensuche?** Hat man eine nützliche Gerade, verschiebt man sie parallel, bis sie einen Pfosten berührt (Abstand genau `r`). Dabei bleibt sie nützlich, denn jeder Pfosten hat
auf der einen Seite der Geraden einen Partner auf der anderen, und die Reihenfolge der Schnittpunkte kann sich nur ändern, wenn zwei Schnittpunkte zusammenfallen, was bei Toren, die sich nicht schneiden, nicht
geschieht. Dann dreht man sie um den Kreis um diesen Pfosten, bis sie einen zweiten Pfosten berührt. So ist jede Lösung zu einer Tangente an zwei Kreise verschiebbar.

**Der Fehler „Tore kürzen“:** Man könnte versuchen, jedes Tor an beiden Enden um `r` zu kürzen und dann nur nach einer Geraden zu suchen, die die gekürzten Tore schneidet (also den Fall `r = 0` zu nehmen). Das ist
falsch: Bei schräg liegenden Geraden ist der kleinste Abstand des Ballmittelpunkts zu einem Pfosten nicht der Abstand des Schnittpunkts mit dem Tor zum Pfosten. Die Lösungshinweise zeigen dazu ein Bild.

**Laufzeit:** Brute Force über die Tangenten braucht etwa `n³` Schritte. Mit einem **Sweep-Verfahren** (man dreht eine Gerade um den Kreis eines Pfostens und hält bei jedem anderen Pfosten an) geht es in
`n² log n`. Das ist aufwendiger zu programmieren.

**Rechnen mit Kommazahlen:** Wir haben die Toleranz `DELTA` gewählt, damit der Ball keinen Pfosten berührt. Ein anderer Weg wäre, ganz mit Brüchen (`fractions.Fraction`) zu rechnen. Das wäre exakt, aber die Wurzeln bei den
gekreuzten Tangenten gehen damit nicht. Dann müsste man die Geraden auf andere Art beschreiben.

**Andere Fragen:** Wenn der Ball im ersten Tor starten darf, braucht er den Abstand `r` dort nicht einzuhalten. Mit `n` Toren in der Ebene liegen die Pfosten oft so, dass die Lösung sehr eindeutig ist, wie bei `krocket5`. Man
könnte auch fragen, mit **wie wenigen Schlägen** es geht, oder wie groß der Ball höchstens sein darf.

**MiniZinc:** Hier haben wir keinen Abschnitt. Es geht um reine Geometrie mit Kommazahlen, nicht um ganzzahlige Entscheidungen.